# Week 05. 동적 페이지 크롤링 개념 재구성

5주차는 Selenium 같은 브라우저 자동화가 필요한 이유를 이해하는 주차입니다.
실제 브라우저 실행은 환경 차이로 실패하기 쉬우므로, 이 노트북은 동적 로딩 전/후 HTML 스냅샷을 사용해 같은 개념을 안전하게 재현합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. 동적 페이지의 핵심 문제

처음 받은 HTML에는 데이터가 없고, 시간이 지난 뒤 JavaScript가 데이터를 채우는 페이지가 있습니다.
아래 두 스냅샷은 로딩 전과 로딩 후를 단순화한 예입니다.


In [1]:
from bs4 import BeautifulSoup
import pandas as pd

loading_html = '<div id="app"><p class="loading">loading...</p></div>'
loaded_html = '''
<div id="app">
  <ul class="webtoons">
    <li data-rank="1"><strong>Orbit Class</strong><span>Kim Writer</span></li>
    <li data-rank="2"><strong>Debug Life</strong><span>Lee Artist</span></li>
    <li data-rank="3"><strong>Async Days</strong><span>Park Studio</span></li>
  </ul>
</div>
'''

print("로딩 전 항목 수:", len(BeautifulSoup(loading_html, "html.parser").select(".webtoons li")))
print("로딩 후 항목 수:", len(BeautifulSoup(loaded_html, "html.parser").select(".webtoons li")))


로딩 전 항목 수: 0
로딩 후 항목 수: 3


## 2. 명시적 대기 로직을 함수로 표현

Selenium의 `WebDriverWait`은 특정 요소가 나타날 때까지 기다립니다.
여기서는 HTML 스냅샷 목록을 순서대로 확인하는 함수로 같은 아이디어를 표현합니다.


In [2]:
def wait_for_selector(html_snapshots, selector):
    for step, html in enumerate(html_snapshots, start=1):
        soup = BeautifulSoup(html, "html.parser")
        found = soup.select(selector)
        if found:
            return step, soup
    raise ValueError(f"selector not found: {selector}")


step, final_soup = wait_for_selector([loading_html, loaded_html], ".webtoons li")
print(f"{step}번째 스냅샷에서 데이터가 나타났습니다.")


2번째 스냅샷에서 데이터가 나타났습니다.


## 3. 로딩 완료 후 데이터 추출

동적 크롤링도 최종적으로는 HTML에서 태그를 골라 데이터를 뽑는 작업입니다.


In [3]:
webtoon_rows = []
for item in final_soup.select(".webtoons li"):
    webtoon_rows.append(
        {
            "rank": int(item["data-rank"]),
            "title": item.select_one("strong").get_text(strip=True),
            "author": item.select_one("span").get_text(strip=True),
        }
    )

webtoons = pd.DataFrame(webtoon_rows)
webtoons


,rank,title,author
0,1,Orbit Class,Kim Writer
1,2,Debug Life,Lee Artist
2,3,Async Days,Park Studio


## 정리

- 동적 페이지는 데이터가 처음 HTML에 없을 수 있습니다.
- 그래서 브라우저 자동화에서는 원하는 요소가 생길 때까지 기다립니다.
- 기다린 뒤에는 BeautifulSoup과 같은 방식으로 구조화된 데이터를 만들 수 있습니다.
